# TransferZAI — BGE-base Fine-Tuning with Hard Negatives

Trains `BAAI/bge-base-en-v1.5` (109M params) on transfer equivalency pairs using TripletLoss with BGE-mined hard negatives.

Improvements over the previous bge-small model:
1. **Larger model** — bge-base has 3x more parameters
2. **Hard negatives** — mines same-catalog confusable courses instead of random in-batch negatives

**Runtime:** GPU (T4 is fine). Runtime → Change runtime type → T4 GPU.

**Output:** Downloads `finetuned_bge_base.zip` at the end.

In [ ]:
!pip install -q sentence-transformers pandas scikit-learn

In [ ]:
import os

# Verify uploaded files
required = [
    'vccs_wm_merged.csv', 'vccs_vt_merged.csv', 'ccc_ucsc_clean.csv',
    'wm_courses_2025.csv', 'vt_courses_2025.csv', 'ucsc_courses_2025.csv',
]
for f in required:
    print(f'  {"✓" if os.path.exists(f) else "✗ MISSING"}  {f}')

In [ ]:
import re, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

QUERY_PREFIX = 'Represent this course for finding transfer equivalents: '
HARD_NEGS_PER_POS = 3

In [ ]:
# ── Helpers ──────────────────────────────────────────────────
_CCC_NOISE = re.compile(
    r'may be offered in a distance.learning format'
    r'|transfer information:|transferable to (?:both )?uc'
    r'|limitations on enrollment:|requisites:'
    r'|minimum credit units|maximum credit units'
    r'|toggle (?:additional|general|learning)'
    r'|grade options:|see open sections|connect with an academic counselor'
    r'|some of the class hours for this course',
    re.IGNORECASE
)
_LAB_CODE  = re.compile(r'[A-Z]+\s+\d+L$')
_LAB_TEXT  = re.compile(r'\b(lab|laboratory)\b', re.I)
_SEQ_CODE  = re.compile(r'([A-Z]+)\s+\d+([A-D])L?$')
_SEQ_TITLE = [
    (1, re.compile(r'\b[1I]\b\s*$')),
    (2, re.compile(r'\b(?:2|II)\b\s*$')),
    (3, re.compile(r'\b(?:3|III)\b\s*$')),
    (4, re.compile(r'\b(?:4|IV)\b\s*$')),
]
_SEQ_DESC  = [
    (2, re.compile(r'\bcontinuation\s+of\b', re.I)),
    (1, re.compile(r'\bfirst\s+(?:course|quarter|semester|part)\b', re.I)),
    (2, re.compile(r'\bsecond\s+(?:course|quarter|semester|part)\b', re.I)),
    (3, re.compile(r'\bthird\s+(?:course|quarter|semester|part)\b', re.I)),
    (3, re.compile(r'\bfinal\s+(?:course|quarter|semester|part)\b', re.I)),
]
_SEQ_LABELS = {1: 'first in sequence', 2: 'second in sequence',
               3: 'third in sequence', 4: 'fourth in sequence'}

def clean_text(text):
    if pd.isna(text) or str(text) in ('Description not found', 'nan', ''): return ''
    text = str(text)
    m = _CCC_NOISE.search(text)
    if m: text = text[:m.start()]
    text = text.lower()
    text = re.sub(r'prerequisite\(s\):.*?(?=\.\s|$)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'cross-listed with:.*?(?=\.\s|$)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def seq_token_from_code(code):
    m = _SEQ_CODE.match(str(code).strip())
    if not m: return None
    return _SEQ_LABELS.get({'A':1,'B':2,'C':3,'D':4}.get(m.group(2)))

def seq_token_from_text(title, desc=''):
    for pos, pat in _SEQ_TITLE:
        if pat.search(title.strip()): return _SEQ_LABELS[pos]
    for pos, pat in _SEQ_DESC:
        if pat.search(str(desc)): return _SEQ_LABELS[pos]
    return None

def lab_token_candidate(code, title, desc):
    if _LAB_CODE.match(str(code).strip()) and (_LAB_TEXT.search(title) or _LAB_TEXT.search(desc)):
        return 'laboratory course'
    return None

def lab_token_query(title):
    return 'laboratory course' if _LAB_TEXT.search(title) else None

def augment(text, *tokens):
    for t in tokens:
        if t: text = f'{text} {t}'
    return text.strip()

def candidate_text(code, title, desc):
    return augment(f'{title} {clean_text(desc)}', seq_token_from_code(code), lab_token_candidate(code, title, desc))

def vccs_query_text(title, desc):
    return augment(f'{title} {clean_text(desc)}', seq_token_from_text(title, desc))

def ccc_query_text(title, desc):
    return augment(f'{title} {clean_text(desc)}', seq_token_from_text(title, desc), lab_token_query(title))

def load_catalog(filename, encoding='utf-8'):
    df = pd.read_csv(filename, encoding=encoding).dropna(subset=['course_code'])
    lk = {}
    for _, r in df.iterrows():
        code = str(r['course_code']).strip()
        lk[code] = {'title': str(r.get('course_title', '')),
                    'description': str(r.get('course_description', '')) if pd.notna(r.get('course_description')) else ''}
    return lk

def parse_vccs(raw):
    parts = re.split(r'\s*TAKEN WITH\s*', str(raw).strip(), flags=re.IGNORECASE)
    courses = []
    for p in parts:
        m = re.match(r'([A-Z]{2,4})\s+(\d{3})\s+(.*)', p.strip())
        if m: courses.append({'dept': m.group(1), 'title': m.group(3).strip()})
    return courses

print('Helpers defined.')

In [ ]:
# ── Load catalogs ────────────────────────────────────────────
wm_lk   = load_catalog('wm_courses_2025.csv', encoding='latin-1')
vt_lk   = load_catalog('vt_courses_2025.csv')
ucsc_lk = load_catalog('ucsc_courses_2025.csv')
print(f'W&M: {len(wm_lk)} | VT: {len(vt_lk)} | UCSC: {len(ucsc_lk)}')

In [ ]:
# ── Load base BGE-base and embed catalogs for hard negative mining ──
print('Loading BAAI/bge-base-en-v1.5 for hard negative mining...')
base_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

def embed_catalog(lk):
    codes = list(lk.keys())
    texts = [candidate_text(c, lk[c]['title'], lk[c]['description']) for c in codes]
    embs  = base_model.encode(texts, batch_size=128, normalize_embeddings=True, show_progress_bar=True)
    return codes, texts, embs

print('Embedding W&M catalog...')
wm_codes, wm_texts, wm_embs = embed_catalog(wm_lk)
print('Embedding VT catalog...')
vt_codes, vt_texts, vt_embs = embed_catalog(vt_lk)
print('Embedding UCSC catalog...')
ucsc_codes, ucsc_texts, ucsc_embs = embed_catalog(ucsc_lk)
print('Done embedding.')

In [ ]:
# ── Mine hard negatives + build triplets ─────────────────────
def mine_hard_negatives(query_text, target_code, catalog_codes, catalog_embs, n=HARD_NEGS_PER_POS):
    q_emb  = base_model.encode([f'{QUERY_PREFIX}{query_text}'], normalize_embeddings=True)[0]
    sims   = catalog_embs @ q_emb
    ranked = np.argsort(sims)[::-1]
    negs   = [catalog_codes[i] for i in ranked if catalog_codes[i] != target_code]
    return negs[:n]

triplets = []   # InputExample(texts=[query, positive, negative])

# W&M
df_wm = pd.read_csv('vccs_wm_merged.csv')
df_wm.columns = df_wm.columns.str.strip()
has = df_wm['W&M Course Code'].notna() & (df_wm['W&M Course Code'].str.strip() != '')
print(f'Mining W&M hard negatives ({has.sum()} pairs)...')
for _, row in df_wm[has].iterrows():
    target = str(row['W&M Course Code']).strip()
    if target not in wm_lk: continue
    courses = parse_vccs(row['VCCS Course'])
    titles  = ' '.join(c['title'] for c in courses)
    query   = vccs_query_text(titles, row.get('VCCS Description', ''))
    pos     = candidate_text(target, wm_lk[target]['title'], wm_lk[target]['description'])
    for neg_code in mine_hard_negatives(query, target, wm_codes, wm_embs):
        triplets.append(InputExample(texts=[f'{QUERY_PREFIX}{query}', pos,
                                            candidate_text(neg_code, wm_lk[neg_code]['title'], wm_lk[neg_code]['description'])]))

# VT
df_vt = pd.read_csv('vccs_vt_merged.csv')
print(f'Mining VT hard negatives ({len(df_vt)} pairs)...')
for _, row in df_vt.iterrows():
    target = row['VT Course Code'].split('+')[0].strip()
    if target not in vt_lk: continue
    courses = parse_vccs(row['VCCS Course'])
    titles  = ' '.join(c['title'] for c in courses)
    query   = vccs_query_text(titles, row.get('VCCS Description', ''))
    vt_title = str(row.get('VT Course Title', ''))
    vt_desc  = str(row.get('VT Description', '')).split('|')[0]
    pos      = candidate_text(target, vt_title, vt_desc)
    for neg_code in mine_hard_negatives(query, target, vt_codes, vt_embs):
        triplets.append(InputExample(texts=[f'{QUERY_PREFIX}{query}', pos,
                                            candidate_text(neg_code, vt_lk[neg_code]['title'], vt_lk[neg_code]['description'])]))

# CCC → UCSC
df_ccc = pd.read_csv('ccc_ucsc_clean.csv')
df_ccc.columns = df_ccc.columns.str.strip()
print(f'Mining UCSC hard negatives ({len(df_ccc)} pairs)...')
for _, row in df_ccc.iterrows():
    target = str(row['UCSC Course Code']).strip()
    if target not in ucsc_lk: continue
    title_raw = str(row.get('CCC Course', '')).strip()
    m = re.match(r'^[A-Z][A-Z0-9 ]*?\s+\S+\s+(.*)', title_raw)
    title = m.group(1).strip() if m else title_raw
    query = ccc_query_text(title, row.get('CCC Description', ''))
    pos   = candidate_text(target, str(row.get('UCSC Course Title', '')), str(row.get('UCSC Description', '')))
    for neg_code in mine_hard_negatives(query, target, ucsc_codes, ucsc_embs):
        triplets.append(InputExample(texts=[f'{QUERY_PREFIX}{query}', pos,
                                            candidate_text(neg_code, ucsc_lk[neg_code]['title'], ucsc_lk[neg_code]['description'])]))

random.shuffle(triplets)
print(f'Total triplets: {len(triplets)}')

In [ ]:
# ── Fine-tune BGE-base with TripletLoss ──────────────────────
# Re-load fresh (not the mining checkpoint)
print('Loading fresh BAAI/bge-base-en-v1.5 for fine-tuning...')
model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

loader   = DataLoader(triplets, shuffle=True, batch_size=32)
loss_fn  = losses.TripletLoss(model)
EPOCHS   = 3
warmup   = int(len(loader) * EPOCHS * 0.1)
print(f'Fine-tuning: {EPOCHS} epochs, {len(loader)} steps/epoch, warmup={warmup}')

model.fit(
    train_objectives=[(loader, loss_fn)],
    epochs=EPOCHS,
    warmup_steps=warmup,
    output_path='./finetuned_bge_base',
    show_progress_bar=True,
)
print('Saved to ./finetuned_bge_base/')

In [ ]:
# ── Download ─────────────────────────────────────────────────
from google.colab import files
!zip -r finetuned_bge_base.zip finetuned_bge_base/
files.download('finetuned_bge_base.zip')
print('Done. Unzip into your transferzaidemo/ project root.')